# Reproducing Liu, Huang, Shen, Shao, Jing et al. (2026), Fig. 5d-i, from dandiset 001751

Paper. Liu R.-F., Huang M., Shen Y., Shao M., Jing J. et al. (2026), "An atlas of primate
insular cortex reveals a signal-processing strategy in von Economo neurons", *Nature Cell
Biology* 28, 1742-1757, doi:10.1038/s41556-026-02009-4 (named on the archive "Cell Atlas and
Novel Signal Processing Strategy in Primate Insular Cortex").

Dataset. [DANDI:001751](https://dandiarchive.org/dandiset/001751/0.260825.0222), version
0.260825.0222 (doi:10.48324/dandi.001751/0.260825.0222), "Electrophysiological Dataset of Local
Neuronal Microcircuits in the Primate Anterior Insular Cortex": the multi-patch recordings of
Fig. 5d-i. The paper cites no version. The archive lists this version and the draft, with the
same number of files and bytes (301 files, 6,964,257,080 bytes, both modified 2026-08-25, printed
by the first code cell), so it is the most recent published version and the one pinned.

Text read. The claims file was written from the version of record (Europe PMC,
PMC13457102): main text, legends, Methods, Data and Code availability. Cell types and the
annotated connections are not in the dandiset; they come from the paper's code,
`metadata_multiPatch.xlsx` in github.com/RFLiu2021/AIC_proj at commit 44cd8a6, matched through
the file names it lists (claim `code-table-link`).

| Claim | Tested here |
|---|---|
| code-table-link (added by the author) | tested |
| fig5d-connectivity-totals | tested |
| fig5-pc-to-pc | tested |
| fig5-ven-outputs-absent | tested |
| fig5-pc-to-ven | tested |
| fig5-pc-in | tested |
| fig5-in-to-ven-absent | tested |
| fig5i-ven-sparse-local-connectivity | tested |
| fig5fg-example-slice | tested |
| fig5-pair-distances | blocked: no soma positions (data missing) |
| fig5-pc-rate-comparable-primate | blocked: the criterion blocks it (methods underspecified); the rate is printed |
| fig4j-eepsp-amplitude (data elsewhere, the code's table) | tested |
| fig3c-ais-distance-length (data elsewhere, the code's table) | blocked: no slice or animal per cell; a variant is shown |
| fig5b-psc-frequency, fig5c-psc-amplitude (data elsewhere, the code's per-cell tables) | tested |
| fig4d-sodium-current (data elsewhere, DANDI:001750) | tested |
| fig3e-phase-peak-index (data elsewhere, the code's table) | tested |
| fig8a-metabolite-gradient, fig8de-glycolytic-bias (data elsewhere, MetaboLights MTBLS13927) | tested |
| patchseq-mapping, ven-map-et-ct, fig6f-g-ven-subtype-mapping (data elsewhere, the code's tables) | tested; patchseq-mapping blocked for its inhibitory half |
| fig4j-eepsp-latency (data elsewhere, the code's table) | blocked: the table holds no latencies |
| 12 other claims with data elsewhere (scRNA-seq, morphology, DANDI:001746, modelling) | blocked: not attempted in this run; one of them, ven-firing-rate-variability, is also blocked by its criterion |

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

import multipatch as mp

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
t_start = t0 = time.time()
assets = mp.asset_table()
recs = mp.read_all(assets.url.tolist(), workers=12)
rec = dict(zip(assets.file, recs))
table = mp.code_table()
print(f"{len(assets)} files, {assets['size'].sum() / 1e9:.2f} GB streamed in {time.time() - t0:.0f} s")
# The versions of the two dandisets used, and whether each draft differs from the version pinned
for d in ["001751", "001750"]:
    vs = mp.requests.get(f"https://api.dandiarchive.org/api/dandisets/{d}/versions/", timeout=60).json()["results"]
    print(f"DANDI:{d} versions:", "; ".join(f"{v['version']} ({v['asset_count']} files, {v['size']:,} bytes, modified {v['modified'][:10]})" for v in vs))


## Data check

Each file is one connection test: one stimulated cell in one slice, eight channels, sweeps of
two 2 ms pulses (or one) with responses on every channel. Below: counts by subject and slice,
sweeps per file, the stimulus, the stated sampling rates and units, the stimulated electrode
the files name against the cell that fires, and repeated recordings.

In [ ]:
files = assets[["file", "subject", "size"]].copy()
files["session"] = [r["session_id"] for r in recs]
files["session_in_name"] = files.file.str.extract(r"ses-([^_]+)_")[0]
files["n_sweeps"] = [max(c["n_sweeps"] for c in r["channels"].values()) for r in recs]
files["n_channels"] = [len(r["channels"]) for r in recs]
files["pre_fires"] = [mp.presynaptic_channel(r) for r in recs]
files["pre_fires_cell"] = [mp.presynaptic_channel(r, mp.CELL_MAX_MV) for r in recs]
files["stim_electrode"] = [r["stimuli"][0]["electrode"].split("/")[-1] for r in recs]
files["stim_onsets"] = [str(r["stimuli"][0]["onsets"]) for r in recs]
files["stim_widths"] = [str(r["stimuli"][0]["widths"]) for r in recs]
files["stim_max"] = [r["stimuli"][0]["max"] for r in recs]
print("session_id equals the file name's session in every file:", (files.session == files.session_in_name).all())
print("\nFiles and slices by subject:")
print(files.groupby("subject").agg(files=("file", "size"), slices=("session", "nunique")).to_string())
print(f"total slices: {files.session.nunique()}")
print("\nSubject metadata (subject_id, sex, age):", sorted({(r['subject_id'], r['sex'], r['age']) for r in recs}))
print("session_start_time values:", sorted({r['start'] for r in recs}))
print("\nChannels per file:", files.n_channels.value_counts().to_dict())
print("Sweeps per file: min", files.n_sweeps.min(), "median", files.n_sweeps.median(), "max", files.n_sweeps.max(),
      f"; files with more than 20 sweeps: {(files.n_sweeps > 20).sum()} of {len(files)}")
print("\nStimulus onsets (samples):", files.stim_onsets.value_counts().to_dict())
print("Stimulus pulse widths (samples):", files.stim_widths.value_counts().to_dict())
print("Stimulus amplitude (stated unit 'amperes'):", files.stim_max.value_counts().to_dict())
print("Stimulus rate and its stated unit:", sorted({(round(r['stimuli'][0]['rate'], 3), r['stimuli'][0]['rate_unit']) for r in recs}))
print("Response series rate (Hz) and data unit:", sorted({x for r in recs for x in r['response_rate_unit']}))
print("Samples per sweep:", sorted({c['n_samples'] for r in recs for c in r['channels'].values()}))
print("\nStimulated electrode named by the stimulus series:", files.stim_electrode.value_counts().to_dict())
print("Channel that fires on the first pulse:", files.pre_fires.value_counts(dropna=False).sort_index().to_dict())
print("Files whose firing channel is the named electrode:", (files.pre_fires == files.stim_electrode.str[-1].astype(int)).sum())
diff = files[files.pre_fires.fillna(-1) != files.pre_fires_cell.fillna(-1)]
print(f"Firing channel when a pre-pulse mean below {mp.CELL_MAX_MV} mV is also required (variant): differs in {len(diff)} files:",
      [(s, a, b) for s, a, b in zip(diff.session, diff.pre_fires, diff.pre_fires_cell)])

In [ ]:
# Ranges and units of the variables used, and checks of the metadata against the values.
base = np.array([c["base_mv"] for r in recs for c in r["channels"].values()])
vmin = np.array([c["vmin"] for r in recs for c in r["channels"].values()])
vmax = np.array([c["vmax"] for r in recs for c in r["channels"].values()])
print(f"Membrane potential (stated volts, shown in mV): pre-pulse means {base.min():.1f} to {base.max():.1f} mV; "
      f"all samples {vmin.min():.1f} to {vmax.max():.1f} mV")
print(f"Channels with a pre-pulse mean below {mp.CELL_MAX_MV} mV (a cell): {(base < mp.CELL_MAX_MV).sum()} of {len(base)}; "
      f"within 1 mV of 0 (no cell): {(np.abs(base) < 1).sum()}")
cell = base[base < mp.CELL_MAX_MV]
print(f"  of cells, pre-pulse mean within -70 +/- 3 mV: {(np.abs(cell + 70) <= 3).sum()}, within -60 +/- 3 mV: {(np.abs(cell + 60) <= 3).sum()}, "
      f"between -30 and -40 mV: {((cell > -40)).sum()}")
spk_lat = []
z = -mp._ms(mp.KEEP_MS[0])
for f, ch in zip(files.file, files.pre_fires):
    if ch == ch:
        a = rec[f]["channels"][int(ch)]["avg"]
        spk_lat.append(np.argmax(a[z:z + mp._ms(10)]) * 1000 / mp.RATE_HZ)
spk_lat = np.array(spk_lat)
print(f"\nPresynaptic spike peak after pulse onset (averaged trace, {len(spk_lat)} files): median {np.median(spk_lat):.2f} ms, "
      f"range {spk_lat.min():.2f} to {spk_lat.max():.2f} ms")
print("A 50-sample pulse is 2 ms at 25 kHz, the Methods' 2 ms; at the stated 1 Hz it would be 50 s, and a 2 ms spike latency would be 2 s.")
# repeated recordings
dig = {}
for f, r in zip(files.file, recs):
    for ch, c in r["channels"].items():
        if c["base_mv"] < mp.CELL_MAX_MV:
            dig.setdefault(c["digest"], []).append((f, ch))
print("\nIdentical channel recordings (sweep arrays) across files or channels:", sum(len(v) > 1 for v in dig.values()))
rep = files.dropna(subset=["pre_fires"]).groupby(["session", "pre_fires"]).file.apply(list)
rep = rep[rep.str.len() > 1]
print(f"Slices where the same cell was stimulated in more than one file: {len(rep)}")
for (s, ch), fl in rep.items():
    print(f"  {s} channel {int(ch)}: " + ", ".join(f"{x.split('obj-')[1][:-12]} ({files.set_index('file').n_sweeps[x]} sweeps)" for x in fl))

In [ ]:
# Everything a file holds outside the sweeps and the stimulus series, and the values of the
# electrode fields: no group or field carries a soma position, a cell type or a connection label.
import re, collections
paths = collections.Counter()
for r in recs:
    for k, kind in {(re.sub(r"Elec_\d+", "Elec_N", k), kind) for k, kind in r["tree"]}:
        paths[(k, kind)] += 1
print("Groups and datasets outside acquisition/<series> and stimulus/presentation/<series> (Elec_N stands for each electrode), files holding each:")
for (k, kind), n in sorted(paths.items()):
    print(f"  {k} [{kind}]: {n}")
gt = collections.defaultdict(collections.Counter)
for r in recs:
    for k, v in r["general_text"].items():
        gt[k][v[:80]] += 1
print("\nValues of the scalar fields under general/ (outside intracellular_ephys):")
for k, c in sorted(gt.items()):
    print(f"  {k}: {dict(c) if len(c) <= 6 else f'{len(c)} distinct values, e.g. ' + str(list(c)[:3])}")
ev = collections.defaultdict(collections.Counter)
cid_ok = 0; n_el = 0
for r in recs:
    for e, d in r["electrodes"].items():
        n_el += 1
        cid_ok += d.get("cell_id") == f"{r['session_id']}_Elec_{int(e.split('_')[1]):03d}"
        for k, v in d.items():
            if k != "cell_id":
                ev[k][str(v)] += 1
print(f"\nElectrode fields over {n_el} electrodes of {len(recs)} files:")
for k, c in ev.items():
    print(f"  {k}: {dict(c)}")
print(f"  cell_id: '<session>_Elec_<channel, 3 digits>' in {cid_ok} of {n_el}")


### The code's table and its file names

The paper's code table `metadata_multiPatch.xlsx` gives the NWB file name once per presynaptic
block. Ten blocks have no file name; each is matched to the file of its slice whose firing channel
is the block's presynaptic channel and which the table does not list elsewhere. Blocks with no
such file are left out of every count, and their rows are printed.

In [ ]:
n_rows_read = len(table)
link = mp.link_blank_blocks(table, files)
print(f"Blocks without a file name: {len(link)}, holding {link.rows.sum()} of the table's {n_rows_read} rows")
print(link.assign(file=link.file.fillna("none").str.split("_obj").str[0] + "...").to_string(index=False))
unlinked = table[table.filename.isna()]
print(f"\nRows left out (no file in the dandiset): {len(unlinked)}; annotated connections among them: {int(unlinked.ConnType.notna().sum())}")
print(unlinked[["slice", "preSynChannel", "postSynChannel", "preSynCellType", "postSynCellType", "ConnType"]].to_string())
table_full = table.copy()
table = table[table.filename.notna()].reset_index(drop=True)
print(f"Rows kept: {len(table)}")

### Clock check

The stimulus series and the responses carry different stated rates (25 kHz and 1 Hz), so the
shared clock is checked on the data. The presynaptic spike peaks about 2 ms after the pulse onset
in the same sample index (above). As a contrast, for every tested pair the averaged postsynaptic
deflection in 1 to 15 ms after the pulse (against the 49 ms before it) is compared with the same
measure taken 35 ms earlier, where there is no pulse. If the responses are locked to the pulse,
the pairs the code annotates as connected stand out at the pulse and not in the shifted window.

In [ ]:
def window_snr(avg, t0_ms):
    x = mp.lowpass(avg.astype(float))
    z = -mp._ms(mp.KEEP_MS[0]) + mp._ms(t0_ms)
    b = x[z - mp._ms(20): z - mp._ms(1)]
    w = x[z + mp._ms(1): z + mp._ms(15)] - b.mean()
    return np.abs(w).max() / b.std()

table["pulse_snr"] = [window_snr(rec[f]["channels"][c]["avg"], 0) for f, c in zip(table.filename, table.postSynChannel)]
table["shifted_snr"] = [window_snr(rec[f]["channels"][c]["avg"], -35) for f, c in zip(table.filename, table.postSynChannel)]
g = table.groupby("connected")[["pulse_snr", "shifted_snr"]].median()
print("Median peak deflection / baseline s.d. (20 ms baseline):")
print(g.rename(index={False: "pairs not annotated", True: "annotated connections"}).round(2).to_string())
c = table[table.connected]
print(f"Annotated connections with a larger deflection at the pulse than in the shifted window: {(c.pulse_snr > c.shifted_snr).sum()} of {len(c)}")
n = table[~table.connected]
print(f"Pairs not annotated, same: {(n.pulse_snr > n.shifted_snr).sum()} of {len(n)}")

## code-table-link

> The rows of the paper's metadata_multiPatch.xlsx (cell types, tested pairs and annotated
> connections) can be linked to the dandiset's files and channels, and they agree with what the
> files hold.

In [ ]:
fl = files.set_index("file")
table["session"] = table.filename.map(fl.session)
table["cls"] = [mp.pair_class(a, b) for a, b in zip(table.preSynCellType, table.postSynCellType)]
table["post_base"] = [rec[f]["channels"][c]["base_mv"] for f, c in zip(table.filename, table.postSynChannel)]
tfiles = table.drop_duplicates("filename")
in_ds = tfiles.filename.isin(files.file)
print(f"Table: {len(table)} rows (ordered pairs) in {len(tfiles)} files; files in the dandiset: {in_ds.sum()} of {len(tfiles)}")
pre_ok = (tfiles.preSynChannel.values == tfiles.filename.map(fl.pre_fires).values)
print(f"Table's presynaptic channel is the channel that fires: {pre_ok.sum()} of {len(tfiles)} files ({100 * pre_ok.mean():.1f}%)")
pre_ok_cell = (tfiles.preSynChannel.values == tfiles.filename.map(fl.pre_fires_cell).values)
print(f"  variant, with a pre-pulse mean below {mp.CELL_MAX_MV} mV also required: {pre_ok_cell.sum()} of {len(tfiles)} ({100 * pre_ok_cell.mean():.1f}%)")
named = ~tfiles.filename.isin(link.file.dropna()).values
print(f"  files the table names itself (the {(~named).sum()} linked by the firing channel left out): {named.sum()}, in the dandiset {in_ds.values[named].sum()}; "
      f"presynaptic channel fires in {pre_ok[named].sum()} of {named.sum()}; with the -30 mV rule {pre_ok_cell[named].sum()}")
bad = tfiles[~pre_ok]
for f, ch in zip(bad.filename, bad.preSynChannel):
    r = rec[f]["channels"][ch]
    print(f"  {f.split('_obj')[0]} channel {ch}: median peak after the pulse {r['peak_after_pulse']:.1f} mV, "
          f"pre-pulse {r['base_mv']:.1f} mV, firing channel found: {fl.pre_fires[f]}")
post_ok = table.post_base < mp.CELL_MAX_MV
print(f"Table's postsynaptic channel holds a cell: {post_ok.sum()} of {len(table)} pairs ({100 * post_ok.mean():.1f}%)")
for thr in [-20, -40, -50]:
    print(f"  with a threshold of {thr} mV: {(table.post_base < thr).sum()}")
print("  pairs whose postsynaptic channel reads within 1 mV of 0 mV (no cell):",
      ", ".join(f"{s} {a}->{b}" for s, a, b, v in zip(table.session, table.preSynChannel, table.postSynChannel, table.post_base) if abs(v) < 1))
print("\nDandiset files the table does not list:")
for f in files.file[~files.file.isin(table.filename)]:
    print(f"  {f}  ({fl.n_sweeps[f]} sweeps, firing channel {fl.pre_fires[f]})")
dup = table[table.duplicated(["session", "preSynChannel", "postSynChannel"], keep=False)]
print(f"\nOrdered pairs listed more than once: {table.duplicated(['session', 'preSynChannel', 'postSynChannel']).sum()}",
      sorted({(s, a, b) for s, a, b in zip(dup.session, dup.preSynChannel, dup.postSynChannel)}))
cells = pd.concat([table[["session", "preSynChannel", "preSynCellType"]].set_axis(["session", "ch", "type"], axis=1),
                   table[["session", "postSynChannel", "postSynCellType"]].set_axis(["session", "ch", "type"], axis=1)]).drop_duplicates()
two = cells[cells.duplicated(["session", "ch"], keep=False)].sort_values(["session", "ch"])
print(f"Cells given two types: {two.drop_duplicates(['session', 'ch']).shape[0]}:",
      "; ".join(f"{s} ch{c}: " + "/".join(g.type) for (s, c), g in two.groupby(["session", "ch"])))
proj = table[table.connected].copy()
proj["proj_ok"] = [p == f"{a}→{b}" for p, a, b in zip(proj.projection, proj.preSynChannel, proj.postSynChannel)]
print("\nAnnotated connections whose 'projection' label disagrees with its channels:")
print(proj[~proj.proj_ok][["session", "preSynChannel", "postSynChannel", "projection"]].to_string())
nc = table.groupby("session").agg(stated=("slice_nconn", "first"), annotated=("connected", "sum"))
nc = nc.dropna()
print(f"\nSlices whose Nr_Conn disagrees with their annotated connections: {(nc.stated != nc.annotated).sum()} of {len(nc)} with Nr_Conn")
print(nc[nc.stated != nc.annotated].to_string())

In [ ]:
stats = pd.DataFrame([mp.response_stats(rec[f]["channels"][c]["avg"]) for f, c in zip(table.filename, table.postSynChannel)])
for col in stats.columns:
    table[col] = stats[col].values
show = table[table.connected][["session", "preSynChannel", "postSynChannel", "cls", "ConnType", "connQuality",
                                "post_base", "peak_mv", "latency_ms", "base_sd_mv", "snr"]]
print("Averaged response (300 Hz low-pass) of each annotated connection; peak in 1 to 15 ms after the pulse:")
print(show.round(2).to_string())
print(f"\nsigned peak against ConnType: Ext with peak > 0: {((show.ConnType == 'Ext') & (show.peak_mv > 0)).sum()} of {(show.ConnType == 'Ext').sum()}; "
      f"Inh with peak < 0: {((show.ConnType == 'Inh') & (show.peak_mv < 0)).sum()} of {(show.ConnType == 'Inh').sum()}")
print("peak / baseline s.d., median: annotated %.1f, not annotated %.1f" % (show.snr.median(), table[~table.connected].snr.median()))
for th in [5, 10, 20]:
    print(f"  above {th}: annotated {(show.snr > th).sum()} of {len(show)}, not annotated {(table[~table.connected].snr > th).sum()} of {(~table.connected).sum()}")
print(f"peak on the first sample of the window (1.00 ms, at_edge, usually a pulse-locked artefact): annotated {int(show.index.isin(table.index[table.at_edge]).sum())} of {len(show)}, "
      f"not annotated {int((table.at_edge & ~table.connected).sum())} of {(~table.connected).sum()}")

print("\nAnnotation checks, reported without deciding:")
inh_pc = show[(show.ConnType == "Inh") & show.cls.str.startswith("PC-")]
print(f"  'Inh' connections from a PC presynaptic cell: {len(inh_pc)} ({inh_pc.cls.value_counts().to_dict()})")
print("  pre-pulse mean of the postsynaptic cell (mV) by ConnType:")
print(show.groupby("ConnType").post_base.describe()[["count", "min", "50%", "max"]].round(1).to_string())
print(f"  'Inh' with a positive peak: {((show.ConnType == 'Inh') & (show.peak_mv > 0)).sum()}; 'Ext' with a negative peak: {((show.ConnType == 'Ext') & (show.peak_mv < 0)).sum()}")
print(show[((show.ConnType == "Inh") & (show.peak_mv > 0)) | ((show.ConnType == "Ext") & (show.peak_mv < 0))][["session", "preSynChannel", "postSynChannel", "cls", "ConnType", "post_base", "peak_mv", "snr"]].round(2).to_string())
print("\nAnnotated connections whose projection label names other channels: response on the channel the label names")
for _, r in proj[~proj.proj_ok].iterrows():
    lab_pre, lab_post = [int(x) for x in r.projection.split("→")]
    cand = files[(files.session == r.session) & (files.pre_fires == lab_pre)].file.tolist()
    f_lab = cand[0] if cand else None
    for name, f, ch in [("table's channels", r.filename, int(r.postSynChannel)),
                        ("label's postsynaptic channel, in the table's file", r.filename, lab_post),
                        ("label's channels", f_lab, lab_post)]:
        if f is None:
            print(f"  {r.session} {r.preSynChannel}->{r.postSynChannel} labelled {r.projection}, {name}: no file whose firing channel is {lab_pre}")
            continue
        s = mp.response_stats(rec[f]["channels"][ch]["avg"])
        print(f"  {r.session} {r.preSynChannel}->{r.postSynChannel} labelled {r.projection}, {name} (pre {fl.pre_fires[f]}, post {ch}): "
              f"peak {s['peak_mv']:+.3f} mV at {s['latency_ms']:.2f} ms, ratio {s['snr']:.2f}, pre-pulse {rec[f]['channels'][ch]['base_mv']:.1f} mV")

**Result: the link holds.** The table names 288 files, and every one is in the dandiset. Ten
presynaptic blocks of the table name no file: 8 are matched one to one to a file of their slice
by the firing channel, and 2 (190315s2 channel 7 and 20241014HS2 channel 8, 13 rows, no annotated
connection) have no file in the dandiset and are left out. In 286 of the 288 files the table
names (99.3%) the table's presynaptic channel is the one that fires (281, 97.6%, when a pre-pulse
mean below -30 mV is also required); the 8 linked files match by construction, which gives 294
of 296 (289 with the -30 mV rule). In the other 2 the table's channel does not overshoot 0 mV
(median peaks -21.5 and -8.0 mV). In 1,372 of 1,403 pairs (97.8%) the postsynaptic channel holds a cell;
6 pairs of slice 190314s1 name channels 6 and 7, which read 0 mV. The dandiset holds 5 files the
table does not list (slice 20240912HS2). Read block by block, the table lists no ordered pair
twice and gives 5 cells two types. Two annotated connections carry a projection label that names
another postsynaptic channel, and in both that channel shows the larger deflection (ratio 47.16
against 2.83, and 17.35 against 8.49), though on the first sample of the window. Nr_Conn
disagrees with the annotations in 4 of 53 slices. The annotated connections stand out in the
averaged traces (median peak / s.d. 8.7 against 2.8), but 333 of 1,377 pairs not annotated exceed
5, so the traces do not reproduce the annotation by this measure. Among the annotations, 8 "Inh"
connections come from a PC, 6 of 12 "Inh" have a positive peak, and the "Inh" postsynaptic cells
sit at -67.6 to -78.4 mV (median -69.9), not at the -60 mV the Methods give for uIPSPs.

## Counts of pairs, cells and connections

The readings the criteria name. Pairs: every row of the table (an ordered pair), and each
ordered pair once. Connections: (a) the dataset's own labels, which do not exist (the files
carry no connection or cell-type label, printed in the data check: electrodes hold only
`cell_id`, `description`, `filtering`, `location`, `resistance`, `seal`, `slice`); (b) the
table's annotations, with and without "OK" quality. Neurons: cells in the table (each slice and
channel once), and channels holding a cell in the dandiset's files (each slice and channel
once), with the threshold varied.

In [ ]:
print("Electrode fields in the files:", sorted({k for r in recs for e in r["electrodes"].values() for k in e}))
uniq = table.drop_duplicates(["session", "preSynChannel", "postSynChannel"])
conn_all = table[table.connected]
conn_noOK = conn_all[conn_all.connQuality != "OK"]
n_cells_table = cells.drop_duplicates(["session", "ch"]).shape[0]
def ds_cells(thr):
    n = 0
    for s, g in files.groupby("session"):
        n += len({ch for f in g.file for ch, c in rec[f]["channels"].items() if c["base_mv"] < thr})
    return n
print(f"Pairs tested: table rows linked to a file {len(table)}, unique ordered pairs {len(uniq)}, all table rows {len(table_full)} (paper 1,292, tolerance 129)")
print(f"Neurons: cells in the table {n_cells_table} (types: {cells.drop_duplicates(['session', 'ch']).type.value_counts().to_dict()}); "
      f"channels holding a cell in the files: " + ", ".join(f"{ds_cells(t)} at {t} mV" for t in [-20, -30, -40, -50]) + " (paper 377, tolerance 38)")
ds_pairs = sum(len({ch for ch, c in rec[f]["channels"].items() if c["base_mv"] < mp.CELL_MAX_MV} - {int(p)})
               for f, p in zip(files.file, files.pre_fires) if p == p)
print(f"Ordered pairs from the files alone (firing channel to every other channel holding a cell): {ds_pairs}")
print(f"Connections: reading (b) all annotated {len(conn_all)}, without 'OK' quality {len(conn_noOK)} (paper 31, tolerance 3.1); reading (a) none, no labels")
print(f"Subjects in the table: {table.Subject_id.nunique()}, slices {table.session.nunique()}")

**Result: not reproduced.** The connection count is 26 under the only reading that can run (23
without "OK" quality), against 31 within 3.1. The pairs tested are within tolerance (1,403 table
rows linked to a file, each a different ordered pair, or 1,416 with the 13 rows that have no file,
against 1,292 within 129). The two neuron counts disagree: the table's cells are 298, outside 377
within 38, while the channels holding a cell in the files are 410 at -30 mV (397 to 411 over the
thresholds printed), inside. The outcome does not depend on which neuron count is used, since the
connection count fails.

In [ ]:
CLASSES = ["PC-to-PC", "VEN-to-VEN", "VEN-to-PC", "PC-to-VEN", "VEN-to-IN", "IN-to-VEN", "PC-to-IN", "IN-to-PC", "IN-to-IN"]
PAPER = {"PC-to-PC": (24, 503), "VEN-to-VEN": (0, 110), "VEN-to-PC": (0, 200), "PC-to-VEN": (1, 200),
         "VEN-to-IN": (0, 55), "IN-to-VEN": (0, 55), "PC-to-IN": (2, 65), "IN-to-PC": (4, 65), "IN-to-IN": (None, None)}
def by_class(t):
    rows = []
    for k in CLASSES:
        g = t[t.cls == k]
        rows.append(dict(cls=k, pairs=len(g), conn=int(g.connected.sum()),
                         conn_noOK=int((g.connected & (g.connQuality != "OK")).sum()),
                         conn_ext=int((g.ConnType == "Ext").sum()),
                         paper_conn=PAPER[k][0], paper_pairs=PAPER[k][1]))
    d = pd.DataFrame(rows).set_index("cls")
    d["rate_%"] = (100 * d.conn / d.pairs).round(2)
    return d
counts_rows = by_class(table)
counts_uniq = by_class(uniq)
counts_cell = by_class(uniq[uniq.post_base < mp.CELL_MAX_MV])
print("All table rows linked to a file:"); print(counts_rows.to_string())
table_full["cls"] = [mp.pair_class(a, b) for a, b in zip(table_full.preSynCellType, table_full.postSynCellType)]
counts_full = by_class(table_full)
print(f"\nVariant, all {len(table_full)} table rows, with the {len(table_full) - len(table)} whose block has no file in the dandiset:")
print(counts_full[["pairs", "conn", "paper_pairs", "rate_%"]].to_string())
print("\nUnique ordered pairs:"); print(counts_uniq.to_string())
print("\nUnique ordered pairs whose postsynaptic channel holds a cell:"); print(counts_cell.to_string())
print("\nBy subject (unique ordered pairs: connections / pairs):")
print(uniq.groupby(["Subject_id", "cls"]).agg(c=("connected", "sum"), n=("connected", "size"))
      .apply(lambda r: f"{int(r.c)}/{r.n}", axis=1).unstack(fill_value="0/0")[CLASSES].to_string())

## fig5-pc-to-pc

> PCs form monosynaptic connections with other PCs in 24 of 503 tested pairs (4.77%).

In [ ]:
for name, d in [("rows", counts_rows), ("unique", counts_uniq), ("postsynaptic cell", counts_cell)]:
    r = d.loc["PC-to-PC"]
    print(f"{name}: {r.conn} of {r.pairs} ({100 * r.conn / r.pairs:.2f}%); without OK {r.conn_noOK} "
          f"({100 * r.conn_noOK / r.pairs:.2f}%); excitatory only {r.conn_ext} ({100 * r.conn_ext / r.pairs:.2f}%)")
print("Paper: 24 of 503 (4.77%); tolerances 2.4, 50 and 0.48 points")

**Result: not reproduced.** The table annotates 18 PC-to-PC connections (11 excitatory, 7
inhibitory) in 743 ordered pairs (749 with the rows that have no file), a rate of 2.42%, against
24 of 503 (4.77%). All three values are outside tolerance under every reading.

## fig5-ven-outputs-absent

> VENs form no connections with adjacent VENs (0 of 110 pairs), with PCs (0 of 200) or with
> interneurons (0 of 55).

The criterion asks for the averaged response of the one VEN-to-PC connection the table
annotates (slice 190315s2, VEN on channel 4 to PC on channel 6).

In [ ]:
print(counts_uniq.loc[["VEN-to-VEN", "VEN-to-PC", "VEN-to-IN"]].to_string())
print("(rows reading:)"); print(counts_rows.loc[["VEN-to-VEN", "VEN-to-PC", "VEN-to-IN"], ["pairs", "conn"]].to_string())
v = table[(table.session == "190315s2") & (table.preSynChannel == 4)]
print("\nSlice 190315s2, presynaptic channel 4 (file " + v.filename.iloc[0] + "):")
print(v[["postSynChannel", "preSynCellType", "postSynCellType", "ConnType", "connQuality", "post_base", "peak_mv", "latency_ms", "base_sd_mv", "snr"]].round(2).to_string())
print("Types the table gives the cells of this slice:", cells[cells.session == "190315s2"].sort_values("ch")[["ch", "type"]].values.tolist())
f = v.filename.iloc[0]
tt = (np.arange(len(rec[f]["channels"][4]["avg"])) / mp.RATE_HZ * 1000) + mp.KEEP_MS[0]
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(tt, rec[f]["channels"][4]["avg"]); ax[0].set(title="ch4 (VEN), presynaptic", xlabel="ms from pulse", ylabel="mV", xlim=(-20, 60))
y = mp.lowpass(rec[f]["channels"][6]["avg"].astype(float))
ax[1].plot(tt, y); ax[1].set(title="ch6 (PC), averaged, 300 Hz low-pass", xlabel="ms from pulse", xlim=(-20, 60))
w = (tt >= -20) & (tt <= 60)
print(f"plotted ranges: ch4 {rec[f]['channels'][4]['avg'][w].min():.1f} to {rec[f]['channels'][4]['avg'][w].max():.1f} mV; ch6 {y[w].min():.2f} to {y[w].max():.2f} mV")
plt.tight_layout(); plt.show()

**Result: partially reproduced.** VEN-to-VEN (0 of 59) and VEN-to-IN (0 of 54) have no
connection, as the paper reports, but the table annotates one VEN-to-PC connection (1 of 146),
and its averaged trace shows a depolarization of 1.01 mV at 12.9 ms after the VEN's spike (peak /
s.d. 95.8), so this reading counts it. The VEN-to-VEN and VEN-to-PC pair counts are outside
tolerance (59 against 110 within 11, 146 against 200 within 20); VEN-to-IN, 54 against 55, is
inside. The absence holds for two of the three partner classes.

## fig5-pc-to-ven

> PCs rarely project to VENs, with one connection in 200 tested PC-to-VEN pairs (0.5%).

In [ ]:
print(counts_uniq.loc[["PC-to-VEN"]].to_string()); print(counts_rows.loc[["PC-to-VEN"], ["pairs", "conn"]].to_string())
print(table[table.connected & (table.cls == "PC-to-VEN")][["session", "preSynChannel", "postSynChannel", "ConnType", "connQuality", "peak_mv", "snr"]].round(2).to_string())

**Result: partially reproduced.** The table annotates one PC-to-VEN connection, as the paper
reports, but in 139 pairs (145 with the rows that have no file), outside 200 within 20 (a rate
of 0.72% against 0.5%).

## fig5-pc-in

> PCs and interneurons are connected in 2 of 65 PC-to-IN pairs (3.08%) and 4 of 65 IN-to-PC
> pairs (6.15%).

In [ ]:
print(counts_uniq.loc[["PC-to-IN", "IN-to-PC"]].to_string()); print(counts_rows.loc[["PC-to-IN", "IN-to-PC"], ["pairs", "conn"]].to_string())
print(table[table.connected & table.cls.isin(["PC-to-IN", "IN-to-PC"])][["session", "preSynChannel", "postSynChannel", "cls", "ConnType", "connQuality", "peak_mv", "snr"]].round(2).to_string())

**Result: partially reproduced.** The connection counts match (2 PC-to-IN, 4 IN-to-PC; without
"OK" quality 1 and 3), but the pairs tested are 83 and 86 (84 and 86 with the rows that have no
file), outside 65 within 6.5.

## fig5-in-to-ven-absent

> Interneurons make no connections onto VENs (0 of 55 pairs).

In [ ]:
print(counts_uniq.loc[["IN-to-VEN"]].to_string()); print(counts_rows.loc[["IN-to-VEN"], ["pairs", "conn"]].to_string())

**Result: reproduced.** No IN-to-VEN connection is annotated, in 54 pairs (55 within 5.5).

## fig5i-ven-sparse-local-connectivity

> VENs have sparse synaptic interactions with the local circuit compared with PCs.

In [ ]:
d = counts_uniq
rate = lambda k: 100 * d.loc[k, "conn"] / d.loc[k, "pairs"]
for k, exp in [("PC-to-PC", 4.77), ("PC-to-IN", 3.08), ("PC-to-VEN", 0.5), ("IN-to-PC", 6.15)]:
    print(f"{k}: {rate(k):.2f}% (paper {exp}%, tolerance {0.1 * exp:.2f})")
print()
for v, p in [("VEN-to-VEN", "PC-to-PC"), ("VEN-to-PC", "PC-to-PC"), ("PC-to-VEN", "PC-to-PC"), ("VEN-to-IN", "PC-to-IN"), ("IN-to-VEN", "IN-to-PC")]:
    tab = [[d.loc[v, "conn"], d.loc[v, "pairs"] - d.loc[v, "conn"]], [d.loc[p, "conn"], d.loc[p, "pairs"] - d.loc[p, "conn"]]]
    odds, pval = fisher_exact(tab)
    print(f"{v} {rate(v):.2f}% against {p} {rate(p):.2f}%: lower {rate(v) < rate(p)}; Fisher exact p = {pval:.3f} (variant)")
print("\nRows reading:", {k: round(100 * counts_rows.loc[k, 'conn'] / counts_rows.loc[k, 'pairs'], 2) for k in ["PC-to-PC", "PC-to-IN", "PC-to-VEN", "IN-to-PC"]})

**Result: partially reproduced.** Every rate with a VEN at either end is below the PC rate of the
same partner class, but none of the four Fig. 5i rates is within tolerance (2.42, 2.41, 0.72 and
4.65% against 4.77, 3.08, 0.5 and 6.15%), and no Fisher test of a VEN rate against its PC rate
gives p below 0.05 (0.159 to 0.636).

## fig5fg-example-slice

> In the example slice of Fig. 5d-h (six simultaneously recorded excitatory neurons), PC cell 2
> evokes EPSPs in cell 6, and VEN cell 4 evokes no EPSP in any of the other cells.

Cells 1 to 6 are channels 1, 2, 3, 6, 7 and 8 of slice 190315s4 (Fig5fg_plotConn.m). The file
for cell 2 is the one whose firing channel is 2, for cell 4 the one whose firing channel is 6.
The `.dat` files hold several test rounds and the code plots round 2; the NWB files carry one set
of sweeps per file with no round, so all sweeps are averaged.

In [ ]:
CELL_OF_CH = {1: 1, 2: 2, 3: 3, 6: 4, 7: 5, 8: 6}
ex = files[files.session == "190315s4"]
fig, axes = plt.subplots(2, 6, figsize=(14, 4), sharex=True)
for row, pre_ch in enumerate([2, 6]):
    f = ex[ex.pre_fires == pre_ch].file.iloc[0]
    print(f"cell {CELL_OF_CH[pre_ch]} (channel {pre_ch}) presynaptic: {f}, {fl.n_sweeps[f]} sweeps (legend: more than 20)")
    t = table[table.filename == f]
    for i, (ch, cn) in enumerate(CELL_OF_CH.items()):
        a = rec[f]["channels"][ch]["avg"].astype(float)
        y = a if ch == pre_ch else mp.lowpass(a)
        tt = np.arange(len(y)) / mp.RATE_HZ * 1000 + mp.KEEP_MS[0]
        w = (tt >= -10) & (tt <= 50)
        axes[row, i].plot(tt[w], y[w] - (0 if ch == pre_ch else y[(tt > -50) & (tt < -1)].mean()), lw=0.8)
        axes[row, i].set_title(f"pre {CELL_OF_CH[pre_ch]} / cell {cn}", fontsize=8)
        if ch != pre_ch:
            s = mp.response_stats(rec[f]["channels"][ch]["avg"])
            r = t[t.postSynChannel == ch]
            ann = r.ConnType.iloc[0] if len(r) and r.connected.iloc[0] else "-"
            typ = r.postSynCellType.iloc[0] if len(r) else "?"
            print(f"   to cell {cn} (ch {ch}, {typ}): annotated {ann}; peak {s['peak_mv']:+.3f} mV at {s['latency_ms']:.2f} ms{' (first sample of the window)' if s['at_edge'] else ''}, "
                  f"baseline s.d. {s['base_sd_mv']:.3f} mV, ratio {s['snr']:.1f}; plotted range {(y[w] - y[(tt > -50) & (tt < -1)].mean()).min():.2f} to {(y[w] - y[(tt > -50) & (tt < -1)].mean()).max():.2f} mV")
plt.tight_layout(); plt.show()
print("Table types for the six cells:", cells[(cells.session == "190315s4")].sort_values("ch")[["ch", "type"]].values.tolist())
print("Annotated pairs from cell 4 (channel 6):", int(table[(table.session == "190315s4") & (table.preSynChannel == 6)].connected.sum()))

**Result: reproduced.** The table annotates cell 2 to cell 6 (channel 2 to channel 8) as an
excitatory connection, and no pair from cell 4; the table's types for the six cells are those of
Fig5e_plotConnexampleFPs.m. Cell 6's averaged response to cell 2 has the largest peak / s.d. of
cell 2's targets (25.1, +0.162 mV at 8.24 ms). The file of cell 4 holds 9 sweeps, fewer than the
legend's "more than 20". From cell 4, cell 5 shows a deflection of +0.511 mV (ratio 30.6) on the
first sample of the window (1.00 ms), the time of the presynaptic spike, which a monosynaptic
EPSP's latency does not fit and a pulse-locked artefact does; cell 2's responses in cells 1 and 3
also peak on that sample (ratios 4.0 and 10.0). The other four responses to cell 4 have ratios
of 1.9 to 2.4. The traces do not decide the claim.

## fig5-pair-distances (blocked)

> PC-PC connections span a wide range of soma distances, 111.8 ± 61.2 um ...

The files hold no soma positions (every group and field of the files and the electrode values are
printed in the data check, and the fields that could hold one below), and neither does the
code's table (its columns printed below). Blocked as data_missing, as the criterion says.

In [ ]:
print("Table columns:", list(mp.code_table().columns))
pos_words = re.compile(r"position|coord|location|distance|depth|xyz|_x$|_y$|plane|image", re.I)
hits = sorted({k for (k, kind) in paths if pos_words.search(k)})
print("Groups or fields of the files whose name suggests a position:", hits)
print("Values those fields hold:", {k: dict(c) for k, c in ev.items() if pos_words.search(k)})
print("Groups under analysis/, processing/, lab_meta_data, intervals/ or general/ other than devices, subject and intracellular_ephys:",
      sorted({k for (k, kind) in paths if re.match(r"(analysis|processing|intervals|general/lab_meta_data|general/(?!devices|subject|intracellular_ephys))", k)}))

## fig5-pc-rate-comparable-primate (blocked)

The criterion blocks this claim as methods_underspecified (the paper prints neither the
reference rate nor what "comparable" allows) and asks for the PC-to-PC rate to be reported.

In [ ]:
r = counts_uniq.loc["PC-to-PC"]
print(f"PC-to-PC rate here: {r.conn} of {r.pairs} = {100 * r.conn / r.pairs:.2f}% (paper 4.77%)")

## Claims on data elsewhere: the paper's code tables

Two claims name a table of the paper's code as their data (`data: elsewhere`), which can be read
here: `sumData_StimWM.xlsx` (eEPSP amplitudes of paired VENs and PCs, Fig. 4j) and
`AIScmp_PCvsVEN_Fig5c.xlsx` (AIS distance and length, Fig. 3c), both at commit 44cd8a6. They do
not use the dandiset.

In [ ]:
import io, requests
RAW = "https://raw.githubusercontent.com/RFLiu2021/AIC_proj/" + mp.CODE_COMMIT + "/"
w = pd.read_excel(io.BytesIO(requests.get(RAW + "ephysio/data_ephy/sumData_StimWM.xlsx", timeout=120).content), sheet_name=0)
print("sumData_StimWM.xlsx, first sheet, columns:", list(w.columns)); print(w.head(8).to_string())
x = w[["stim current (mA)", "EPSP_amp_PC", "EPSP_amp_VEN"]].copy()
x["pair"] = x["stim current (mA)"].isna().cumsum()
x = x.dropna()
x["pair"] = pd.factorize(x.pair)[0]
print(f"Block 'copied data': {x.pair.nunique()} pairs; stimulation levels per pair: {x.groupby('pair').size().value_counts().to_dict()}; "
      f"levels {sorted(x['stim current (mA)'].unique())} (stated mA)")
print(f"Amplitudes (unit not stated): PC {x.EPSP_amp_PC.min():.4f} to {x.EPSP_amp_PC.max():.4f}, VEN {x.EPSP_amp_VEN.min():.4f} to {x.EPSP_amp_VEN.max():.4f}")
long = x.melt(id_vars=["pair", "stim current (mA)"], value_vars=["EPSP_amp_PC", "EPSP_amp_VEN"], var_name="cell", value_name="amp")
long["cell"] = long.cell.str[-3:].str.strip("_")
long = long.rename(columns={"stim current (mA)": "level"})
print("\nMedian amplitude by level:"); print(long.pivot_table(index="level", columns="cell", values="amp", aggfunc="median").to_string())
print(f"VEN above PC in {(x.EPSP_amp_VEN > x.EPSP_amp_PC).sum()} of {len(x)} pair-levels; per pair (mean over levels): "
      f"{(x.groupby('pair').EPSP_amp_VEN.mean() > x.groupby('pair').EPSP_amp_PC.mean()).sum()} of {x.pair.nunique()}")
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm, AnovaRM
ols = smf.ols("amp ~ C(cell) * C(level)", data=long).fit()
print("\n(a) ordinary two-way ANOVA (type II):"); print(anova_lm(ols, typ=2).round(5).to_string())
rm = AnovaRM(long, "amp", "pair", within=["cell", "level"]).fit()
print("\n(b) repeated-measures two-way ANOVA, pair as subject:"); print(rm.anova_table.round(5).to_string())

**Result: reproduced.** In the table's 23 pairs (the paper states 26), VEN amplitudes are
higher than PC amplitudes at every stimulation level (medians printed above; 20 of 23 pairs on
the mean over levels), and the cell-type effect has p = 0.0056 in the ordinary two-way ANOVA and
p = 0.0065 in the repeated-measures ANOVA (paper 0.0034); the stimulation level effect is
significant under both.

In [ ]:
a = pd.read_excel(io.BytesIO(requests.get(RAW + "morpho/dataMorph/AIScmp_PCvsVEN_Fig5c.xlsx", timeout=120).content), sheet_name=0, header=None)
hrow = [i for i in range(len(a)) if (a.iloc[i] == "PC_AIS_L (um)").any()][0]
cols = {v: i for i, v in a.iloc[hrow].items() if isinstance(v, str)}
print("Header row:", list(cols))
vals = {k: pd.to_numeric(a.iloc[hrow + 1:, i], errors="coerce").dropna().values for k, i in cols.items()}
for k, v in vals.items():
    print(f"  {k}: n = {len(v)}, median {np.median(v):.2f}, range {v.min():.2f} to {v.max():.2f}")
print("The table has no slice or animal column, so the two-way ANOVA readings of the criterion cannot be run.")
from scipy.stats import mannwhitneyu
for m in ["AIS_D", "AIS_L"]:
    pc, ven = vals[f"PC_{m} (um)"], vals[f"VEN_{m} (um)"]
    print(f"Variant, two-sided Wilcoxon rank-sum, {m}: VEN median {np.median(ven):.2f} against PC {np.median(pc):.2f}, p = {mannwhitneyu(ven, pc).pvalue:.2g}")
print("Paper: n = 108 VENs and 430 PCs")

**Result: blocked.** The table holds 120 VEN and 500 PC values (the paper states 108 and 430)
without the slice or animal of each cell, so the criterion's two-way ANOVA cannot be run. As a
variant that does not decide, the rank-sum test puts the VEN AIS farther from the soma (median
15.85 against 3.95 um) and shorter (24.33 against 29.13 um), both with p below 1e-15.

## fig5b-psc-frequency and fig5c-psc-amplitude (data elsewhere, the code's tables)

> The frequencies of spontaneous and miniature EPSCs and IPSCs are lower in VENs than in
> neighbouring PCs. / The amplitudes ... do not differ significantly between VENs and PCs.

The recordings are in DANDI:001752. The per-cell event frequencies and amplitudes the paper's
code tabulated for Fig. 5b,c (`sPSCs_Fig7ab.xlsx`, sheet summary, and `miniEPSCs_Fig7ab.xlsx`,
sheets mEPSC_sum and mIPSC_sum, commit 44cd8a6) are tested here; the events are not detected
again from the recordings.

In [ ]:
from scipy.stats import mannwhitneyu
def xl(path, **kw):
    return pd.read_excel(io.BytesIO(requests.get(RAW + path, timeout=120).content), header=None, **kw)
sp = xl("ephysio/data_ephy/sPSCs_Fig7ab.xlsx", sheet_name="summary")
def block(rows, cols):
    b = sp.iloc[rows, cols].dropna()
    return pd.DataFrame({"freq": pd.to_numeric(b.iloc[:, 0]), "amp": pd.to_numeric(b.iloc[:, 1])})
psc = {
    "sEPSC": (block(slice(3, 14), [8 - 6, 3]), block(slice(3, 14), [11, 12])),
    "sIPSC": (block(slice(21, 29), [2, 3]), block(slice(21, 29), [11, 12])),
}
mi = xl("ephysio/data_ephy/miniEPSCs_Fig7ab.xlsx", sheet_name="mEPSC_sum")
mven = mi.iloc[1:9, [1, 2]].astype(float).set_axis(["freq", "amp"], axis=1)
mpc = mi.iloc[1:9, [5, 6]].astype(float).set_axis(["freq", "amp"], axis=1)
psc["mEPSC"] = (mpc, mven)
mip = xl("ephysio/data_ephy/miniEPSCs_Fig7ab.xlsx", sheet_name="mIPSC_sum").iloc[1:].dropna(subset=[1])
mip = mip.set_axis(["cell", "type", "n", "dur", "freq", "amp"], axis=1)
psc["mIPSC"] = (mip[mip.type == "PC"][["freq", "amp"]].astype(float), mip[mip.type == "VEN"][["freq", "amp"]].astype(float))
legend_n = {"sEPSC": (11, 8), "sIPSC": (8, 7), "mEPSC": (8, 8), "mIPSC": (21, 15)}
rows = []
for k, (pc, ven) in psc.items():
    for m in ["freq", "amp"]:
        rows.append(dict(type=k, measure=m, n_PC=len(pc), n_VEN=len(ven), legend_n_PC=legend_n[k][0], legend_n_VEN=legend_n[k][1],
                         median_PC=pc[m].median(), median_VEN=ven[m].median(),
                         p=mannwhitneyu(ven[m], pc[m], alternative="two-sided").pvalue))
psc_res = pd.DataFrame(rows)
print("Units as tabulated: frequency in Hz; amplitude unscaled for sEPSC, sIPSC and mEPSC (the mEPSC sheet notes a factor of 100 to pA), pA for mIPSC.")
print(psc_res.round(4).to_string())

**Result: fig5b reproduced, fig5c reproduced.** The cell counts are the legend's for all four
event types. The VEN median frequency is lower than the PC median for every type, with
two-sided rank-sum p of 0.0068 (sEPSC), 0.0093 (sIPSC), 0.0281 (mEPSC) and 0.0102 (mIPSC); the
amplitudes give p of 0.1087, 0.6943, 0.5737 and 0.1399, none below 0.05.

## fig4d-sodium-current (data elsewhere, DANDI:001750)

> Somatic Na+ currents in outside-out patches are lower in VENs (n = 14) than in L5 PCs (n = 21).

The 35 files of DANDI:001750 (version 0.260825.0222; the archive lists it and a draft with the same files and bytes, printed by the first code cell) are streamed;
the cell type of each comes from the code's `metadata_outSideOutSodiumCurrentRec.csv` (commit
44cd8a6), matched by file name, which also lists a `max_Na_current` per patch, the values
`Fig4d_plotNaCurrent.m` tests. Steps run from a holding of -80 mV, 25 ms long (Methods, "families
of Na+ currents in response to a series of depolarizing voltages ranging from -80 to 0 mV (25 ms)
with 5-mV stepping"; leak subtracted online); `Fig4d_plotNaCurrent.m` also uses the steps up to
0 mV ("test pulses from -80 to 0 mV") and a baseline over the 2 ms before the step. Reading (a)
of the criterion, the largest peak inward current over the steps, is taken from the sweep
average of each step. The Methods give neither a filter nor the window after the step, so the
measure is checked against the code's own per-patch values: a list of filters, baselines and
windows is run, and each is compared with `max_Na_current`. Reading (b), per area from
capacitance, cannot run (the files hold no capacitance); since the Methods take every patch to
be about 5 um2 (50 fF at 1 uF/cm2), it would divide every patch by the same area, which changes
neither the order of the values nor the rank-sum test.

In [ ]:
import collections
from scipy.signal import butter, filtfilt
with mp.DandiAPIClient() as client:
    na_assets = sorted((a.path, a.get_content_url(follow_redirects=1, strip_query=True))
                       for a in client.get_dandiset("001750", "0.260825.0222").get_assets() if a.path.endswith(".nwb"))
def na_read(url):
    """Sweep average per step (A), keyed by (step level in V, onset sample)."""
    import h5py
    f = h5py.File(io.BytesIO(requests.get(url, timeout=600).content), "r")
    acq, st = f["acquisition"], f["stimulus/presentation"]
    by = collections.defaultdict(list); rate = None
    for n in acq:
        d = acq[n]["data"][:] * acq[n]["data"].attrs["conversion"]
        rate = float(acq[n]["starting_time"].attrs["rate"])
        sv = st[n.replace("NaCurrentSeries", "VoltageClampStimulusSeries")]["data"][:]
        ch = np.flatnonzero(np.diff(sv) != 0)
        if len(ch) == 0:
            continue
        on = ch[0] + 1
        by[(round(float(sv[on]), 3), on)].append(d)
    return {k: np.mean(v, axis=0) for k, v in by.items()}, rate, len(acq)
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(8) as ex:
    na = list(ex.map(na_read, [u for _, u in na_assets]))
na_meta = pd.read_csv(io.BytesIO(requests.get(RAW + "ephysio/data_ephy/rawdata_ephysio/metadata_outSideOutSodiumCurrentRec.csv", timeout=120).content))
na_files = [p.split("/")[-1] for p, _ in na_assets]
print(f"{len(na_files)} files; in the code's table: {sum(f in set(na_meta.fileName_nwb) for f in na_files)}; types {na_meta.cellType.value_counts().to_dict()}")
print("sampling rates (Hz):", sorted({round(r) for _, r, _ in na}), "; sweeps per file:", sorted({n for _, _, n in na}))
print("step onsets (samples):", sorted({k[1] for s, _, _ in na for k in s}), "; step levels (V):", sorted({k[0] for s, _, _ in na for k in s})[:3], "...", sorted({k[0] for s, _, _ in na for k in s})[-3:])
FS = 100_000   # Methods, "collected at a 100-kHz sample frequency"; checked above
def na_peak(steps, t0_ms, t1_ms, vmax, fc=None, base="code"):
    """Largest inward current (pA) and its time (ms after the step) over steps up to vmax.
    fc: low-pass cutoff (2nd-order Butterworth, forward and backward), None for none.
    base: 'code', the 2 ms before the step (Fig4d_plotNaCurrent.m); 'long', 5 to 0.1 ms before."""
    if fc:
        b, a = butter(2, fc / (FS / 2))
    best, tbest = 0.0, np.nan
    for (v, on), x in steps.items():
        if v > vmax + 1e-9:
            continue
        y = filtfilt(b, a, x) if fc else x
        y0 = y[on - 200:on].mean() if base == "code" else y[on - 500:on - 10].mean()
        seg = (y[on + int(t0_ms * 100): on + int(t1_ms * 100)] - y0) * 1e12
        j = int(np.argmin(seg))
        if seg[j] < best:
            best, tbest = float(seg[j]), t0_ms + j / 100
    return best, tbest
typ = dict(zip(na_meta.fileName_nwb, na_meta.cellType))
code_val = dict(zip(na_meta.fileName_nwb, na_meta.max_Na_current))
ty = np.array([typ[f] for f in na_files]); cv = np.array([code_val[f] for f in na_files])
rows = []
for fc in [None, 500, 1000, 2000]:
    for base in (["long", "code"] if fc is None else ["code"]):
        for t0, t1 in [(0.2, 3), (0.5, 3), (0.3, 5), (0.3, 10)]:
            for vmax in [0.0, 0.085]:
                pk = np.array([na_peak(s, t0, t1, vmax, fc, base)[0] for s, _, _ in na])
                rows.append(dict(lowpass_Hz=fc or "none", baseline=base, window_ms=f"{t0}-{t1}", steps_up_to_mV=int(vmax * 1000),
                                 median_VEN_pA=np.median(pk[ty == "VEN"]), median_PC_pA=np.median(pk[ty == "PC"]),
                                 p=mannwhitneyu(pk[ty == "VEN"], pk[ty == "PC"]).pvalue, r_with_code=np.corrcoef(pk, cv)[0, 1],
                                 median_abs_diff_code_pA=np.median(np.abs(pk - cv))))
na_res = pd.DataFrame(rows)
pd.set_option("display.max_rows", 100)
print("\nReading (a), per filter, baseline, window and step range (inward current negative):"); print(na_res.round(4).to_string())
cv_v, cv_p = cv[ty == "VEN"], cv[ty == "PC"]
print(f"\nReference, the code's max_Na_current: VEN median {np.median(cv_v):.2f}, PC {np.median(cv_p):.2f} pA, rank-sum p = {mannwhitneyu(cv_v, cv_p).pvalue:.4f} (n = {len(cv_v)} and {len(cv_p)})")
for lab, fc in [("unfiltered", None), ("1 kHz low-pass", 1000)]:
    s = na_res[(na_res.lowpass_Hz == (fc or "none")) & (na_res.baseline == "code") & (na_res.steps_up_to_mV == 0)]
    print(f"{lab}, code's baseline, steps up to 0 mV: p from {s.p.min():.4f} to {s.p.max():.4f} over the four windows; r with the code from {s.r_with_code.min():.2f} to {s.r_with_code.max():.2f}")
for lab, fc in [("unfiltered", None), ("1 kHz low-pass", 1000)]:
    s = na_res[(na_res.lowpass_Hz == (fc or "none")) & (na_res.baseline == "code") & (na_res.steps_up_to_mV == 85)]
    print(f"{lab}, code's baseline, steps up to +85 mV: p from {s.p.min():.4f} to {s.p.max():.4f}; r with the code from {s.r_with_code.min():.2f} to {s.r_with_code.max():.2f}")
for m_ in ["ephysio/Fig4d_plotNaCurrent.m", "ephysio/extFig9ad_plotVENsubtypeAP_Cmp.m"]:
    code_m = requests.get(RAW + m_, timeout=120).text.splitlines()
    print(f"Lines of the code's {m_.split('/')[-1]} that call a filter:", [f"{k + 1}: {l.strip()}" for k, l in enumerate(code_m) if "filter" in l.lower()])
main = [na_peak(s, 0.2, 3, 0.0, 1000, "code") for s, _, _ in na]
raw = [na_peak(s, 0.2, 10, 0.0, None, "code") for s, _, _ in na]
per = pd.DataFrame(dict(file=[f.split("_obj-")[1][:8] for f in na_files], type=ty, code_pA=cv,
                        lp1k_pA=[m[0] for m in main], lp1k_t_ms=[m[1] for m in main],
                        raw_0_2_10_pA=[r[0] for r in raw], raw_t_ms=[r[1] for r in raw]))
print("\nPer patch: the code's value; the 1 kHz low-pass peak (0.2 to 3 ms) and its time; the unfiltered peak (0.2 to 10 ms) and its time")
print(per.round(2).to_string())
print(f"time of the unfiltered peak: later than 3 ms in {(per.raw_t_ms > 3).sum()} of {len(per)} patches; 1 kHz peak time median {per.lp1k_t_ms.median():.2f} ms, range {per.lp1k_t_ms.min():.2f} to {per.lp1k_t_ms.max():.2f}")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
for t, c in [("VEN", "tab:red"), ("PC", "tab:blue")]:
    m = per.type == t
    ax[0].scatter(per.code_pA[m], per.lp1k_pA[m], c=c, s=15, label=t)
    ax[1].scatter(per.code_pA[m], per.raw_0_2_10_pA[m], c=c, s=15, label=t)
for a_, lab in zip(ax, ["1 kHz low-pass, 0.2-3 ms", "unfiltered, 0.2-10 ms"]):
    lim = [min(per.code_pA.min(), per.lp1k_pA.min(), per.raw_0_2_10_pA.min()) - 2, 2]
    a_.plot(lim, lim, "k:", lw=0.8); a_.set(xlabel="code's max_Na_current (pA)", ylabel="peak measured here (pA)", title=lab, xlim=lim, ylim=lim); a_.legend()
plt.tight_layout(); plt.show()
print(f"plotted ranges: code {per.code_pA.min():.1f} to {per.code_pA.max():.1f}, 1 kHz {per.lp1k_pA.min():.1f} to {per.lp1k_pA.max():.1f}, unfiltered {per.raw_0_2_10_pA.min():.1f} to {per.raw_0_2_10_pA.max():.1f} pA; none outside the axes")

**Result: reproduced, resting on an assumption.** The VEN median peak is smaller in magnitude
than the PC median under all 40 readings run. Low-passed at 500 Hz to 2 kHz, the peak gives
p = 0.0006 to 0.0040 under every window and step range. At 1 kHz with the code's 2 ms baseline
and steps up to 0 mV it agrees with the code's own `max_Na_current` (r = 0.99, median difference
1.28 pA, every window; with steps up to +85 mV r is 0.77 to 0.94) and gives VEN -17.3 against PC -31.5 pA, p = 0.0010, the p of the code's values. Unfiltered, the
result depends on the window (p = 0.0067 to 0.0956), the agreement with the code is weaker
(r = 0.40 to 0.90), and 6 of 35 unfiltered peaks fall later than 3 ms (up to 9.99 ms), where the
low-passed peaks fall at 0.59 to 1.42 ms. The 1 kHz low-pass, baseline over the 2 ms before the
step, 0.2 to 3 ms, steps up to 0 mV, is adopted, because it reproduces the code's per-patch values.
Reading (b) divides every patch by the same estimated area and gives the same test.

## fig3e-phase-peak-index (data elsewhere, the code's table)

> VENs have lower AP phase-plane peak indices than every PC type.

`PeakIdx_Fig5e.xlsx` (commit 44cd8a6) lists the peak index per cell by transcriptomic type; its
statistics sheet names Dunn's test with Bonferroni-corrected p-values, which settles the
correction the legend leaves open.

In [ ]:
from scipy.stats import kruskal, rankdata, norm
pk = xl("ephysio/data_ephy/PeakIdx_Fig5e.xlsx", sheet_name="Sheet1")
print("PeakIdx_Fig5e.xlsx, Sheet1, first rows:"); print(pk.head(4).to_string())
groups = {}
for j in range(0, pk.shape[1], 3):
    name = pk.iloc[0, j]
    v = pd.to_numeric(pk.iloc[1:, j + 1], errors="coerce").dropna().values
    groups[name] = v
for k, v in groups.items():
    print(f"{k}: n = {len(v)}, median {np.median(v):.3f}, range {v.min():.3f} to {v.max():.3f}")
print("Fig. 3e legend: cells per class as in Fig. 2d-f, L2/3 IT HPCAL1 30, IT RORB 17, L5 IT FEZF2 119, VEN 22, L5 ET 16, L6 PC 13")
H, pkw = kruskal(*groups.values())
print(f"Kruskal-Wallis H = {H:.2f}, p = {pkw:.2g}")
allv = np.concatenate(list(groups.values())); ranks = rankdata(allv); N = len(allv)
_, tc = np.unique(allv, return_counts=True); tie = (tc ** 3 - tc).sum() / (12 * (N - 1))
idx = np.cumsum([0] + [len(v) for v in groups.values()])
rbar = {k: ranks[idx[i]:idx[i + 1]].mean() for i, k in enumerate(groups)}
npairs = len(groups) * (len(groups) - 1) // 2
print(f"\nDunn's test, VEN against each PC type (Bonferroni over {npairs} pairs):")
for k in groups:
    if k == "VEN":
        continue
    se = np.sqrt((N * (N + 1) / 12 - tie) * (1 / len(groups["VEN"]) + 1 / len(groups[k])))
    z = (rbar["VEN"] - rbar[k]) / se
    p = 2 * norm.sf(abs(z))
    print(f"  VEN against {k}: z = {z:.2f}, p = {p:.2g}, Bonferroni p = {min(1, p * npairs):.2g}")

**Result: reproduced.** The VEN median peak index (0.116, n = 32) is the lowest of the six
types (0.218 to 0.286), the Kruskal-Wallis test gives p = 6.5e-08, and Dunn's test separates VENs
from each of the five PC types with Bonferroni-corrected p from 3.2e-08 to 0.001. The table holds
more cells per type than the legend's counts (VEN 32 against 22, L2/3 IT 33 against 30, IT RORB
24 against 17, L5 IT 124 against 119, L5 ET 20 against 16, L6 PC 16 against 13).

## fig8a-metabolite-gradient and fig8de-glycolytic-bias (data elsewhere, the code's table)

> Of 581 metabolites, 228 differ between AIC and V1 and only 108 between AIC and parietal cortex.
> / Compared with V1, AIC shows elevated citrate, reduced 2-ketobutyrate and increased glutamate.

The data are in MetaboLights MTBLS13927. Its metabolite assignment file (MAF) of revision
MTBLS13927_01, the only revision listed (2026-02-28, printed below), holds the processed intensity of each identified
metabolite in each sample, and is tested here. The paper's code holds the same table,
`VEN_Metabolism/Metabolic_data.xlsx` (sheet Identified, commit 44cd8a6); the two are compared
below.
Welch's ANOVA across the three regions, then Games-Howell for each pair (Fig. 8 legend). The
paper does not say whether intensities were log-transformed, nor whether a pair is counted only
when the ANOVA is also significant; the four combinations are run.

In [ ]:
import pingouin as pg
import hashlib
MAF_URL = ("https://ftp.ebi.ac.uk/pub/databases/metabolights/studies/public/MTBLS13927/METADATA_REVISIONS/MTBLS13927_01/"
           "m_MTBLS13927_LC-MS_alternating_hilic_metabolite_profiling_v2_maf.tsv")
rev_html = requests.get(MAF_URL.split("MTBLS13927_01/")[0], timeout=120).text
print("Revisions of MTBLS13927 under METADATA_REVISIONS (name, modified):", re.findall(r'href="(MTBLS13927_\d+)/".*?(\d{4}-\d\d-\d\d)', rev_html))
for attempt in range(5):   # the EBI server refuses connections at times; retry
    try:
        maf_raw = requests.get(MAF_URL, timeout=300).content
        break
    except requests.ConnectionError:
        time.sleep(30)
print(f"MAF, revision MTBLS13927_01: {len(maf_raw)} bytes, sha256 {hashlib.sha256(maf_raw).hexdigest()}")
maf = pd.read_csv(io.BytesIO(maf_raw), sep="\t")
first = list(maf.columns).index("smallmolecule_abundance_std_error_sub") + 1
maf_samples = list(maf.columns[first:])
print("MAF sample columns:", maf_samples)
# sample names as the code's table spells them ("A1C-1-Q" is AIC 1 Q)
maf = maf.rename(columns={c: c.replace("A1C", "AIC").replace("-", "_") for c in maf_samples})
samples = [c.replace("A1C", "AIC").replace("-", "_") for c in maf_samples]
region = [c.split("_")[0] for c in samples]
print(f"{len(maf)} identified metabolites in the deposit (paper 581), run modes {maf.charge.value_counts().to_dict()}; samples per region:",
      pd.Series(region).value_counts().to_dict(), "(legend 6 AIC, 6 PrC, 7 V1)")
mt = pd.read_excel(io.BytesIO(requests.get(RAW + "VEN_Metabolism/Metabolic_data.xlsx", timeout=120).content), sheet_name="Identified")
print("The code's Metabolic_data.xlsx, sheet Identified, columns:", list(mt.columns))
print(f"The code's Metabolic_data.xlsx: {len(mt)} metabolites; same names in the same order: {(mt['Metabolite Name'].values == maf.metabolite_identification.values).all()}")
A, B = maf[samples].astype(float).values, mt[samples].astype(float).values
same = np.isclose(A, B, rtol=1e-9)
print(f"  intensities equal in {same.sum()} of {same.size}; columns that differ: {[s for s, k in zip(samples, same.all(axis=0)) if not k]}")
print("  V1_4_Q of one equals V1_4_S of the other:", np.allclose(maf["V1_4_Q"].astype(float), mt["V1_4_S"].astype(float)) and np.allclose(maf["V1_4_S"].astype(float), mt["V1_4_Q"].astype(float)),
      "(both are V1 samples, so the region tests are the same on either table)")
mt = mt.assign(runmode=maf.charge.values) if "runmode" not in mt else mt
X = A
print(f"intensity range {np.nanmin(X):.3g} to {np.nanmax(X):.3g}; zeros {(X == 0).sum()}, missing {np.isnan(X).sum()}")
def gh_all(values):
    out = []
    for i, row in enumerate(values):
        df = pd.DataFrame({"y": row, "g": region})
        if df.groupby("g").y.std().min() == 0:
            out.append((np.nan, np.nan, np.nan, np.nan)); continue
        wa = pg.welch_anova(df, dv="y", between="g")
        pa = float(wa[[c for c in wa.columns if c.replace("-", "_") == "p_unc"][0]].iloc[0])
        gh = pg.pairwise_gameshowell(df, dv="y", between="g")
        def pp(a, b):
            r = gh[((gh.A == a) & (gh.B == b)) | ((gh.A == b) & (gh.B == a))]
            return float(r["pval"].iloc[0])
        out.append((pa, pp("AIC", "V1"), pp("AIC", "PrC"), pp("PrC", "V1")))
    return pd.DataFrame(out, columns=["p_anova", "AIC_V1", "AIC_PrC", "PrC_V1"])
res_raw = gh_all(X)
res_log = gh_all(np.log2(X + 1))
for name, r in [("raw", res_raw), ("log2(x+1)", res_log)]:
    for gate in [False, True]:
        ok = (r.p_anova < 0.05) if gate else pd.Series(True, index=r.index)
        print(f"{name:10s} {'ANOVA p<0.05 required' if gate else 'Games-Howell only    '}: AIC-V1 {((r.AIC_V1 < 0.05) & ok).sum()}, AIC-PrC {((r.AIC_PrC < 0.05) & ok).sum()}, PrC-V1 {((r.PrC_V1 < 0.05) & ok).sum()}"
              f"  (untestable {r.p_anova.isna().sum()})")
print("Paper: AIC-V1 228 (tolerance 23), AIC-PrC 108 (tolerance 11)")

In [ ]:
names = mt["Metabolite Name"]
pick = names.str.contains(r"^citric acid$|^citrate$|isocitric|2-Ketobutyric|^L-Glutamic acid$|^Glutamic acid glutamate$", case=False, regex=True)
print("Metabolites matched by name:", names[pick].tolist())
print("Any name containing 'citr':", names[names.str.contains("citr", case=False)].tolist())
for i in np.flatnonzero(pick.values):
    row = X[i]
    med = {g: np.median(row[np.array(region) == g]) for g in ("AIC", "PrC", "V1")}
    print(f"{names.iloc[i]} ({mt.runmode.iloc[i]}): medians AIC {med['AIC']:.4g}, PrC {med['PrC']:.4g}, V1 {med['V1']:.4g}; "
          f"AIC/V1 {med['AIC'] / med['V1']:.2f}; Games-Howell AIC-V1 p raw {res_raw.AIC_V1.iloc[i]:.3g}, log {res_log.AIC_V1.iloc[i]:.3g}; Welch ANOVA p raw {res_raw.p_anova.iloc[i]:.3g}")

**Result: fig8a partially reproduced; fig8de partially reproduced.** The MetaboLights deposit
(revision MTBLS13927_01) holds 558 identified metabolites, not the paper's 581, and its
intensities are those of the code's table, with the labels of two V1 samples (V1_4_Q and V1_4_S)
swapped. More metabolites differ between AIC and V1 than between AIC and PrC under every reading
(175 to 195 against 84 to 98), the gradient the paper reports, but the AIC-V1 count is below 228
within 23 under every reading, and the AIC-PrC count is within 108 within 11 only on log
intensities without the ANOVA gate (98). Compared with V1, citric acid is higher in AIC (ratio
1.42, Games-Howell p = 0.00046) and L-glutamic acid is higher (1.35, p = 0.0001), but
2-ketobutyric acid, lower (0.59), does not reach p < 0.05 (0.18 raw, 0.087 log); the second
glutamate entry, "Glutamic acid glutamate", is higher by 1.23 with p = 0.15.

## Patch-seq mapping: patchseq-mapping, ven-map-et-ct, fig6f-g-ven-subtype-mapping (data elsewhere, the code's tables)

The reads are in GEO GSE319369. The mapping results the paper's code saved,
`patchSeqMappingTo10X/data/mappingRes/results_Ext_240415.csv` (excitatory Patch-seq neurons
mapped to the 20 excitatory leiden clusters) and `mappingResults_neurons.csv` (all neurons), at
commit 44cd8a6, are read here; the names of the excitatory clusters are those that the code's
notebook `mapping2Ins10XClusters.ipynb` (the cell that defines `projNeuronTypeName`, index 56) assigns: leiden 16 is L5 ET and
leiden 8 is L5/6 CT (Exc FEZF2 COL24A1).

In [ ]:
MR = RAW + "patchSeqMappingTo10X/data/mappingRes/"
ext = pd.read_csv(io.BytesIO(requests.get(MR + "results_Ext_240415.csv", timeout=120).content), index_col=0)
allN = pd.read_csv(io.BytesIO(requests.get(MR + "mappingResults_neurons.csv", timeout=120).content), index_col=0)
import json
code_nb = json.loads(requests.get(RAW + "patchSeqMappingTo10X/mapping2Ins10XClusters.ipynb", timeout=120).content)
print("Cell 56 of the code's mapping2Ins10XClusters.ipynb, which names the excitatory leiden clusters:")
print("".join(code_nb["cells"][56]["source"]))
EXT_NAMES = {**{k: "L2/3 IT HPCAL1" for k in (1, 5, 15)}, **{k: "IT RORB" for k in (0, 2, 7, 13, 19)},
             **{k: "L5 IT FEZF2" for k in (3, 6, 9, 11, 14, 18)}, 8: "L5/6 CT", 4: "L6 IT", 16: "L5 ET", 12: "L6b", 17: "L6b", 10: "NP"}
print(f"Excitatory Patch-seq neurons in results_Ext: {len(ext)} (paper 273); distinct excitatory clusters they map to: "
      f"{ext.predictClustersIDs.nunique()} of 20 (paper 18)")
print("cells per cluster:", ext.predictClustersIDs.value_counts().sort_index().to_dict())
rest = allN[~allN.index.isin(ext.index)]
print(f"\nAll-neuron table: {len(allN)} neurons; not in the excitatory table: {len(rest)}, "
      f"their names {rest.patchCellSubtypeNames.value_counts(dropna=False).to_dict()}")
inh_named = rest[~rest.patchCellSubtypeNames.astype(str).str.contains("VEN|PC", case=False)]
print(f"  of them with an inhibitory name (not VEN or PC): {len(inh_named)} (paper 34 inhibitory, tolerance 3.4)")
print("The all-neuron table's cluster ids are those of the joint neuron clustering, not the 33 inhibitory types; no saved table gives the inhibitory mapping.")
ven = ext[ext.patchCellTypeIDs == 2]
print(f"\nVENs (patchCellTypeIDs 2) in the excitatory table: {len(ven)} (paper 29)")
ct = pd.crosstab(ven.patchCellSubtypeNames, ven.predictClustersIDs.map(EXT_NAMES))
print(ct.to_string())
n_vl = int((ven.patchCellSubtypeNames.str.lower() == "ven_long").sum()); n_vs = int((ven.patchCellSubtypeNames.str.lower() == "ven_short").sum())
vl_et = int(((ven.patchCellSubtypeNames.str.lower() == "ven_long") & (ven.predictClustersIDs == 16)).sum())
vs_ct = int(((ven.patchCellSubtypeNames.str.lower() == "ven_short") & (ven.predictClustersIDs == 8)).sum())
print(f"\nven-map-et-ct: L5 ET {int((ven.predictClustersIDs == 16).sum())} (paper 14, tolerance 1.4), L5/6 CT {int((ven.predictClustersIDs == 8).sum())} (paper 15, tolerance 1.5)")
print(f"fig6f-g: VEN-L mapped to L5 ET {vl_et} of {n_vl} VEN-L (paper 10, tolerance 1); VEN-S mapped to L5/6 CT {vs_ct} of {n_vs} VEN-S (paper 11, tolerance 1.1)")
print(f"patchseq-mapping: excitatory neurons {len(ext)} (273, tolerance 27); excitatory types {ext.predictClustersIDs.nunique()} (18, tolerance 1.8); "
      f"inhibitory neurons {len(inh_named)} (34, tolerance 3.4); inhibitory types: not saved (17)")
print(f"\nNotebook run time to here: {(time.time() - t_start) / 60:.1f} min")


**Result: ven-map-et-ct reproduced; fig6f-g-ven-subtype-mapping partially reproduced;
patchseq-mapping blocked.** The 29 VENs map to L5 ET (14, against 14 within 1.4) and L5/6 CT
(15, against 15 within 1.5), as the paper states. Every VEN-L maps to L5 ET and every VEN-S to
L5/6 CT, but there are 14 and 15 of them, outside the paper's 10 within 1 and 11 within 1.1; the
subcluster COL24A1 POC5 is not in the saved results and is not tested. The 273 excitatory neurons
map to 17 of the 20 excitatory clusters (18 within 1.8), and 33 of the other neurons have
inhibitory names (34 within 3.4; 35 with one VEN and one PC_L5). The inhibitory mapping is not
saved, so the 17 of 33 inhibitory types cannot be checked.

## Summary

| Claim | Outcome |
|---|---|
| code-table-link | reproduced |
| fig5d-connectivity-totals | not reproduced |
| fig5-pc-to-pc | not reproduced |
| fig5-ven-outputs-absent | partially reproduced |
| fig5-pc-to-ven | partially reproduced |
| fig5-pc-in | partially reproduced |
| fig5-in-to-ven-absent | reproduced |
| fig5i-ven-sparse-local-connectivity | partially reproduced |
| fig5fg-example-slice | reproduced |
| fig5-pair-distances | blocked (data missing) |
| fig5-pc-rate-comparable-primate | blocked (methods underspecified) |
| fig4j-eepsp-amplitude (elsewhere) | reproduced |
| fig3c-ais-distance-length (elsewhere) | blocked (metadata insufficient) |
| fig5b-psc-frequency (elsewhere) | reproduced |
| fig5c-psc-amplitude (elsewhere) | reproduced |
| fig4d-sodium-current (elsewhere) | reproduced, resting on an assumption |
| fig3e-phase-peak-index (elsewhere) | reproduced |
| fig8a-metabolite-gradient (elsewhere) | partially reproduced |
| fig8de-glycolytic-bias (elsewhere) | partially reproduced |
| ven-map-et-ct (elsewhere) | reproduced |
| fig6f-g-ven-subtype-mapping (elsewhere) | partially reproduced |
| patchseq-mapping (elsewhere) | blocked (metadata insufficient) |
| fig4j-eepsp-latency (elsewhere) | blocked (no latencies in the table) |
| 12 other claims on data elsewhere | blocked (not attempted in this run) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper gives no rule for calling a
connection, so connections are the annotations of the paper's code table, not a detection from
the traces. Cell types are the table's. The Na+ current peak is taken after a 1 kHz low-pass, a
choice that reproduces the code's per-patch values. All sweeps of a file are averaged; the code plots round
2 of each `.dat` file, which the NWB files do not separate.

What was found about the dataset. The files carry no cell types, connection labels or soma
positions, so Fig. 5 cannot be tested from the dandiset alone. The response series state a rate
of 1 Hz, the stimulus rate is given with the unit "millisecond", the stimulus amplitude is 2
"amperes" (2 nA in the Methods), every stimulus series names Elec_8 as its electrode while the
cell that fires is another in most files, and every file has the session start time
2018-01-01. The code's table, the only source of labels, names no file for 10 of its
presynaptic blocks (8 match a file of the dandiset by the firing channel, 2 have none), counts
fewer cells (298) and more pairs (1,403 linked, 1,416 in all) than the paper (377 and 1,292), and
annotates 26 connections, not 31; per class its pair counts differ from the paper's by more than
10% in 6 of 8 classes. Of the other code tables read for claims on
data elsewhere, several hold different numbers of cells or items than the paper states (23 eEPSP
pairs against 26, 120 and 500 AIS values against 108 and 430, 14 and 15 VEN subtypes against 10
and 11, and more cells per type in the peak-index table than the legend states). The MetaboLights
deposit holds the same 558 identified metabolites as the code's table, against the paper's 581.